# Aula 13 - Memória do Agente


## Configuração

Este notebook demonstra como construir um agente de reserva de viagens com **memória persistente** usando o **Microsoft Agent Framework** (MAF).

Você aprenderá como diferentes tipos de memória do agente — de trabalho, de curto prazo e de longo prazo — afetam a forma como um agente retém e utiliza informações ao longo das conversas.

**Pré-requisitos:**
- Um projeto Azure AI Foundry com um modelo de chat implantado (por exemplo, `gpt-4o-mini`).
- Ter feito login com o Azure CLI — execute `az login` no seu terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — o endpoint do seu projeto Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — o nome do seu modelo implantado.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Tipos de Memória do Agente

Agentes de IA podem aproveitar diferentes tipos de memória, cada um servindo a um propósito distinto:

### Memória de Trabalho
O próprio fio da conversa — as mensagens trocadas em uma única sessão. O agente pode se referir a mensagens anteriores no mesmo fio para manter a coerência. No MAF, isso é criado via **`agent.create_session()`**, que retorna uma `AgentSession`.

### Memória de Curto Prazo
Informações que persistem durante a duração de uma tarefa ou sessão, mas que não são armazenadas permanentemente. Por exemplo, o agente pode acumular fatos durante uma conversa de planejamento com múltiplas interações e usá-los para produzir um itinerário final.

### Memória de Longo Prazo
Preferências e fatos que persistem **entre sessões**. Um usuário que retorna não deve ter que repetir suas restrições alimentares ou estilo de viagem. A memória de longo prazo é tipicamente suportada por um armazenamento externo — um banco de dados, arquivo ou índice vetorial — e disponibilizada para o agente via ferramentas.


## Memória de Trabalho com Sessões

A forma mais simples de memória é a sessão de conversa. Quando você passa a mesma sessão (criada via `agent.create_session()`) para chamadas sucessivas de `agent.run()`, o agente vê todo o histórico daquela conversa e pode recordar detalhes anteriores.

Vamos criar um agente de viagens e demonstrar a memória de trabalho.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

O agente lembrou corretamente do orçamento porque ambas as mensagens compartilham a mesma sessão. Isso é **memória de trabalho** — existe apenas durante a vida útil da sessão.

### O que acontece com um novo tópico?

Se criarmos uma sessão **nova**, o agente não tem memória da conversa anterior:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Padrão de Memória de Longo Prazo

Para lembrar as preferências do usuário **entre sessões**, precisamos de um armazenamento persistente que fique fora da thread da conversa. O agente acessa esse armazenamento por meio de **ferramentas** — funções que ele pode chamar para salvar e recuperar informações.

Abaixo implementamos um armazenamento simples de preferências em memória (em produção você usaria um banco de dados ou índice vetorial) e o expomos como ferramentas que o agente pode usar.

### Arquitetura
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Cenário 1 — Usuário de primeira viagem reserva uma viagem de aniversário

Sarah visita pela primeira vez. O agente deve armazenar suas preferências por meio das ferramentas e usá-las para recomendar hotéis.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Cenário 2 — Sarah retorna semanas depois

Sarah inicia um **novo tópico** (simulando uma nova sessão). A memória de trabalho está vazia, mas o armazenamento de preferências de longo prazo ainda possui suas informações. O agente deve recuperá-las e usá-las para personalizar as recomendações.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Resumo

Nesta lição, você aprendeu três tipos de memória de agente e como implementá-los com o Microsoft Agent Framework:

| Tipo de Memória | Mecanismo MAF | Duração |
|---|---|---|
| **Trabalho** | `agent.create_session()` | Conversa única |
| **Curto prazo** | Contexto acumulado dentro de um thread | Tarefa / sessão única |
| **Longo prazo** | Armazenamento externo acessado via funções `@tool` | Entre sessões |

### Principais Lições
1. **`agent.create_session()`** fornece memória de trabalho — o agente vê todo o histórico de conversas dentro de uma sessão.
2. **Novas sessões perdem o contexto** — sem memória de longo prazo, o agente não pode relembrar conversas passadas.
3. **Funções `@tool`** fazem a ponte — permitem que o agente salve e recupere informações de um armazenamento persistente.
4. **A personalização melhora com o tempo** — quanto mais preferências são armazenadas, melhores as recomendações do agente.

### Aplicações no Mundo Real
- **Atendimento ao Cliente**: Lembrar histórico e preferências do cliente
- **Assistentes Pessoais**: Manter contexto ao longo de dias ou semanas
- **Saúde**: Acompanhar informações e preferências do paciente
- **E-commerce**: Compras personalizadas com base no histórico

### Próximos Passos
- Substituir o dict em memória por um banco de dados ou armazenamento vetorial (ex. Azure AI Search)
- Adicionar expiração de memória para informações sensíveis ao tempo
- Construir sistemas multiagente com memória compartilhada
- Explorar o notebook Cognee para memória baseada em grafo de conhecimento


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:  
Este documento foi traduzido utilizando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos para garantir a precisão, fique ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original em seu idioma original deve ser considerado a fonte autorizada. Para informações críticas, é recomendada a tradução profissional feita por humanos. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações equivocadas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
